# To Review the Result For Research Question 2
### For table 3,  Please Click [Tables 4](#table-rq2-3)
### For table 4,  Please Click [Tables 5](#table-rq2-4)
### For table 5,  Please Click [Tables 6](#table-rq2-5)

In [1]:
# ============================================================
# RQ2 Reproduction
# Cell 1: Imports and project paths
# ============================================================

from pathlib import Path
import json
import math

import numpy as np
import pandas as pd

from datasets import Dataset
from scipy.stats import binomtest
from statsmodels.discrete.conditional_models import ConditionalLogit


# ============================================================
# Project root
# ============================================================

PROJECT_ROOT = Path(
    "/home/chufeng/Desktop/CSI/ICSE2027"
)


# ============================================================
# Raw NVIDIA Open-SWE-Traces
# Hugging Face Arrow shards
# ============================================================

NVIDIA_ROOT = (
    PROJECT_ROOT
    / "Data Preparation"
    / "dataset"
    / "open_swe_traces"
)


NVIDIA_GROUPS = {
    "OpenHands + MiniMax-M2.5":
        NVIDIA_ROOT
        / "openhands"
        / "minimax_m25",

    "OpenHands + Qwen3.5-122B":
        NVIDIA_ROOT
        / "openhands"
        / "qwen35_122b",

    "SWE-agent + MiniMax-M2.5":
        NVIDIA_ROOT
        / "sweagent"
        / "minimax_m25",

    "SWE-agent + Qwen3.5-122B":
        NVIDIA_ROOT
        / "sweagent"
        / "qwen35_122b",
}


# ============================================================
# Final TP annotation files
# ============================================================

TP_ROOT = (
    PROJECT_ROOT
    / "Data Preparation"
    / "data"
)


NVIDIA_TP_FILE = (
    TP_ROOT
    / "nvidia_final.jsonl"
)


# ============================================================
# Output
# ============================================================

RQ2_OUTPUT_DIR = (
    TP_ROOT
    / "rq2_final"
)

RQ2_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# Agent / model coding
# ============================================================

CONFIG_INFO = {
    "OpenHands + MiniMax-M2.5": {
        "agent": "OpenHands",
        "model": "MiniMax",
    },

    "OpenHands + Qwen3.5-122B": {
        "agent": "OpenHands",
        "model": "Qwen",
    },

    "SWE-agent + MiniMax-M2.5": {
        "agent": "SWE-agent",
        "model": "MiniMax",
    },

    "SWE-agent + Qwen3.5-122B": {
        "agent": "SWE-agent",
        "model": "Qwen",
    },
}


CONFIG_ORDER = list(
    NVIDIA_GROUPS.keys()
)


/home/chufeng/Desktop/CSI/ICSE2027/.venv/lib/python3.13/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# Cell 2: Path sanity check
# ============================================================

print("=" * 100)
print("RAW NVIDIA DATA")
print("=" * 100)

for config, folder in NVIDIA_GROUPS.items():

    arrow_files = sorted(
        folder.rglob("*.arrow")
    )

    print(
        f"{config:35s} "
        f"exists={folder.exists()}  "
        f"arrow_files={len(arrow_files)}"
    )


print("\n" + "=" * 100)
print("FINAL TP ANNOTATION")
print("=" * 100)

print(
    f"exists={NVIDIA_TP_FILE.exists()}  "
    f"{NVIDIA_TP_FILE.name}"
)

assert NVIDIA_TP_FILE.exists(), (
    f"Combined NVIDIA annotation file not found:\n"
    f"{NVIDIA_TP_FILE}"
)


print(
    f"\nRQ2 output:\n"
    f"{RQ2_OUTPUT_DIR}"
)

RAW NVIDIA DATA
OpenHands + MiniMax-M2.5            exists=True  arrow_files=29
OpenHands + Qwen3.5-122B            exists=True  arrow_files=36
SWE-agent + MiniMax-M2.5            exists=True  arrow_files=32
SWE-agent + Qwen3.5-122B            exists=True  arrow_files=32

FINAL TP ANNOTATION
exists=True  nvidia_final.jsonl

RQ2 output:
/home/chufeng/Desktop/CSI/ICSE2027/Data Preparation/data/rq2_final


In [3]:
# ============================================================
# Cell 3: Load all raw NVIDIA Arrow trajectories
# ============================================================

raw_parts = []


for config, folder in NVIDIA_GROUPS.items():

    print("\n" + "=" * 100)
    print(config)
    print("=" * 100)

    arrow_files = sorted(
        folder.rglob("*.arrow")
    )

    if not arrow_files:

        raise FileNotFoundError(
            f"No Arrow files found:\n{folder}"
        )


    config_parts = []


    for shard_i, arrow_file in enumerate(
        arrow_files,
        start=1,
    ):

        ds = Dataset.from_file(
            str(arrow_file)
        )


        required_columns = {
            "instance_id",
            "trajectory_id",
            "resolved",
        }


        missing = (
            required_columns
            - set(ds.column_names)
        )


        if missing:

            raise ValueError(
                f"{arrow_file} "
                f"missing columns: {missing}"
            )


        shard_df = pd.DataFrame(
            {
                "instance_id":
                    ds[
                        "instance_id"
                    ],

                "trajectory_id":
                    ds[
                        "trajectory_id"
                    ],

                "resolved":
                    ds[
                        "resolved"
                    ],
            }
        )


        shard_df[
            "configuration"
        ] = config


        shard_df[
            "source_shard"
        ] = arrow_file.name


        config_parts.append(
            shard_df
        )


    # --------------------------------------------------------
    # Combine shards for this configuration
    # --------------------------------------------------------

    config_df = pd.concat(
        config_parts,
        ignore_index=True,
    )


    # --------------------------------------------------------
    # Normalize identifiers
    # --------------------------------------------------------

    config_df[
        "instance_id"
    ] = (
        config_df[
            "instance_id"
        ]
        .astype(str)
        .str.strip()
    )


    config_df[
        "trajectory_id"
    ] = (
        config_df[
            "trajectory_id"
        ]
        .astype(str)
        .str.strip()
    )


    config_df[
        "resolved"
    ] = (
        pd.to_numeric(
            config_df[
                "resolved"
            ],
            errors="raise",
        )
        .astype(int)
    )


    # --------------------------------------------------------
    # Basic diagnostics
    # --------------------------------------------------------

    print(
        f"Arrow shards : "
        f"{len(arrow_files):,}"
    )

    print(
        f"Trajectories : "
        f"{len(config_df):,}"
    )

    print(
        f"Unique traj  : "
        f"{config_df['trajectory_id'].nunique():,}"
    )

    print(
        f"Unique PRs   : "
        f"{config_df['instance_id'].nunique():,}"
    )

    print(
        "\nResolved:"
    )

    print(
        config_df[
            "resolved"
        ]
        .value_counts()
        .sort_index()
    )


    duplicated = (
        config_df[
            "trajectory_id"
        ]
        .duplicated()
        .sum()
    )


    print(
        f"\nDuplicate trajectory IDs: "
        f"{duplicated:,}"
    )


    raw_parts.append(
        config_df
    )



OpenHands + MiniMax-M2.5
Arrow shards : 29
Trajectories : 49,948
Unique traj  : 49,948
Unique PRs   : 20,098

Resolved:
resolved
-1    11096
 0    22911
 1    15941
Name: count, dtype: int64

Duplicate trajectory IDs: 0

OpenHands + Qwen3.5-122B
Arrow shards : 36
Trajectories : 55,488
Unique traj  : 55,488
Unique PRs   : 20,362

Resolved:
resolved
-1    12684
 0    29147
 1    13657
Name: count, dtype: int64

Duplicate trajectory IDs: 0

SWE-agent + MiniMax-M2.5
Arrow shards : 32
Trajectories : 57,268
Unique traj  : 57,268
Unique PRs   : 20,791

Resolved:
resolved
-1    13163
 0    24554
 1    19551
Name: count, dtype: int64

Duplicate trajectory IDs: 0

SWE-agent + Qwen3.5-122B
Arrow shards : 32
Trajectories : 44,785
Unique traj  : 44,785
Unique PRs   : 18,211

Resolved:
resolved
-1     9815
 0    18875
 1    16095
Name: count, dtype: int64

Duplicate trajectory IDs: 0


In [4]:
# ============================================================
# Cell 4: Build raw NVIDIA master population
# ============================================================

raw_df = pd.concat(
    raw_parts,
    ignore_index=True,
)


# ------------------------------------------------------------
# Agent / model metadata
# ------------------------------------------------------------

raw_df[
    "agent"
] = (
    raw_df[
        "configuration"
    ]
    .map(
        lambda config:
            CONFIG_INFO[
                config
            ][
                "agent"
            ]
    )
)


raw_df[
    "model"
] = (
    raw_df[
        "configuration"
    ]
    .map(
        lambda config:
            CONFIG_INFO[
                config
            ][
                "model"
            ]
    )
)


print("=" * 100)
print("RAW NVIDIA MASTER POPULATION")
print("=" * 100)

print(
    f"Rows            : "
    f"{len(raw_df):,}"
)

print(
    f"Unique traj IDs : "
    f"{raw_df['trajectory_id'].nunique():,}"
)

print(
    f"Unique PR IDs   : "
    f"{raw_df['instance_id'].nunique():,}"
)


display(
    raw_df.head()
)

RAW NVIDIA MASTER POPULATION
Rows            : 207,489
Unique traj IDs : 207,489
Unique PR IDs   : 22,320


,instance_id,trajectory_id,resolved,configuration,source_shard,agent,model
0,python-attrs__attrs-770,f8853683-87bd-418b-b87f-19ab95c60317,1,OpenHands + MiniMax-M2.5,data-00000-of-00029.arrow,OpenHands,MiniMax
1,altair-viz__altair-3118,4003b9f9-720f-4897-a270-586055e35ab7,0,OpenHands + MiniMax-M2.5,data-00000-of-00029.arrow,OpenHands,MiniMax
2,open-telemetry__opentelemetry-go-contrib-6962,fe108abd-08de-44cb-b75c-1c9a5fd164d0,-1,OpenHands + MiniMax-M2.5,data-00000-of-00029.arrow,OpenHands,MiniMax
3,apache__dubbo-go-hessian2-212,702b0dab-5adb-481d-a410-27be367373f8,0,OpenHands + MiniMax-M2.5,data-00000-of-00029.arrow,OpenHands,MiniMax
4,alibaba__fescar-412,6922fc2a-9d34-4bfa-ab08-d46a37b5928b,0,OpenHands + MiniMax-M2.5,data-00000-of-00029.arrow,OpenHands,MiniMax


In [5]:
# ============================================================
# Cell 5: Raw population summary
# ============================================================

raw_summary = (
    raw_df
    .groupby(
        "configuration",
        as_index=False,
    )
    .agg(
        trajectories=(
            "trajectory_id",
            "size",
        ),

        unique_trajectories=(
            "trajectory_id",
            "nunique",
        ),

        unique_prs=(
            "instance_id",
            "nunique",
        ),
    )
)


display(
    raw_summary.style.format(
        {
            "trajectories":
                "{:,}",

            "unique_trajectories":
                "{:,}",

            "unique_prs":
                "{:,}",
        }
    )
)

,configuration,trajectories,unique_trajectories,unique_prs
0,OpenHands + MiniMax-M2.5,"49,948","49,948","20,098"
1,OpenHands + Qwen3.5-122B,"55,488","55,488","20,362"
2,SWE-agent + MiniMax-M2.5,"57,268","57,268","20,791"
3,SWE-agent + Qwen3.5-122B,"44,785","44,785","18,211"


In [6]:
# ============================================================
# Cell 6: Load final TP trajectory annotations
#
# NVIDIA annotations are stored in one combined file:
#     TP_ROOT / "nvidia_final.jsonl"
#
# Configuration membership is reconstructed from raw_df using
# trajectory_id. Downstream cells continue to use:
#
#     tp_sets[configuration] -> set of TP trajectory IDs
# ============================================================

# ------------------------------------------------------------
# 1. Build raw trajectory_id -> configuration lookup
# ------------------------------------------------------------

trajectory_to_configs = (
    raw_df[
        [
            "trajectory_id",
            "configuration",
        ]
    ]
    .drop_duplicates()
    .groupby("trajectory_id")[
        "configuration"
    ]
    .agg(set)
    .to_dict()
)


# ------------------------------------------------------------
# 2. Resolve configuration for one annotation record
# ------------------------------------------------------------

def infer_annotation_configuration(
    record,
    trajectory_id,
    candidate_configs,
):

    if len(candidate_configs) == 1:
        return next(iter(candidate_configs))


    # If configuration metadata was preserved in the combined file,
    # use it to disambiguate duplicated trajectory IDs.
    for key in (
        "configuration",
        "config",
        "source_configuration",
    ):

        value = record.get(key)

        if value is None:
            continue

        value = str(value).strip()

        if value in candidate_configs:
            return value


    # Fall back to agent/model metadata if present.
    agent_value = str(
        record.get(
            "agent",
            record.get(
                "agent_name",
                "",
            ),
        )
    ).strip().lower()

    model_value = str(
        record.get(
            "model",
            record.get(
                "model_name",
                "",
            ),
        )
    ).strip().lower()


    possible = set(candidate_configs)


    if agent_value:

        if (
            "openhands" in agent_value
            or "open_hands" in agent_value
        ):

            possible = {
                config
                for config in possible
                if config.startswith("OpenHands")
            }

        elif (
            "swe-agent" in agent_value
            or "swe_agent" in agent_value
            or agent_value == "swe"
        ):

            possible = {
                config
                for config in possible
                if config.startswith("SWE-agent")
            }


    if model_value:

        if "minimax" in model_value:

            possible = {
                config
                for config in possible
                if "MiniMax" in config
            }

        elif "qwen" in model_value:

            possible = {
                config
                for config in possible
                if "Qwen" in config
            }


    if len(possible) == 1:
        return next(iter(possible))


    raise ValueError(
        "Cannot uniquely assign combined NVIDIA annotation "
        "to one configuration:\n"
        f"  trajectory_id={trajectory_id}\n"
        f"  candidate_configs={sorted(candidate_configs)}\n"
        f"  record agent={agent_value!r}\n"
        f"  record model={model_value!r}"
    )


# ------------------------------------------------------------
# 3. Read nvidia_final.jsonl once
#
# Canonical key is configuration + trajectory_id.
# If the same key occurs multiple times, the last occurrence wins.
# ------------------------------------------------------------

canonical_by_config = {
    config: {}
    for config in NVIDIA_GROUPS
}

raw_rows_by_config = {
    config: 0
    for config in NVIDIA_GROUPS
}

duplicate_rows_by_config = {
    config: 0
    for config in NVIDIA_GROUPS
}

unmatched_annotation_rows = []


with NVIDIA_TP_FILE.open(
    "r",
    encoding="utf-8",
) as f:

    for line_number, line in enumerate(
        f,
        start=1,
    ):

        line = line.strip()

        if not line:
            continue


        record = json.loads(line)


        trajectory_id = record.get(
            "trajectory_id"
        )

        if trajectory_id is None:
            continue


        trajectory_id = str(
            trajectory_id
        ).strip()


        candidate_configs = (
            trajectory_to_configs.get(
                trajectory_id,
                set(),
            )
        )


        if not candidate_configs:

            unmatched_annotation_rows.append(
                {
                    "line_number":
                        line_number,

                    "trajectory_id":
                        trajectory_id,

                    "classification":
                        record.get(
                            "classification"
                        ),
                }
            )

            continue


        config = infer_annotation_configuration(
            record=record,
            trajectory_id=trajectory_id,
            candidate_configs=candidate_configs,
        )


        raw_rows_by_config[
            config
        ] += 1


        canonical = (
            canonical_by_config[
                config
            ]
        )


        if trajectory_id in canonical:

            duplicate_rows_by_config[
                config
            ] += 1


        # Last occurrence wins.
        canonical[
            trajectory_id
        ] = record


# ------------------------------------------------------------
# 4. Rebuild the original tp_sets[config] interface
# ------------------------------------------------------------

tp_sets = {}

annotation_summary_rows = []


for config in NVIDIA_GROUPS:

    canonical = (
        canonical_by_config[
            config
        ]
    )

    tp_ids = set()

    classification_counts = {}


    for trajectory_id, record in canonical.items():

        classification = str(
            record.get(
                "classification",
                "",
            )
        ).strip().upper()


        classification_counts[
            classification
        ] = (
            classification_counts.get(
                classification,
                0,
            )
            + 1
        )


        if classification == "TEST_PANDERING":

            tp_ids.add(
                trajectory_id
            )


    tp_sets[
        config
    ] = tp_ids


    annotation_summary_rows.append(
        {
            "configuration":
                config,

            "raw_annotation_rows":
                raw_rows_by_config[
                    config
                ],

            "duplicate_rows":
                duplicate_rows_by_config[
                    config
                ],

            "unique_annotation_ids":
                len(canonical),

            "tp_ids":
                len(tp_ids),

            "classification_counts":
                classification_counts,
        }
    )


annotation_summary = pd.DataFrame(
    annotation_summary_rows
)


display(
    annotation_summary
)


# ------------------------------------------------------------
# 5. Combined-file linkage diagnostics
# ------------------------------------------------------------

unmatched_annotation_df = pd.DataFrame(
    unmatched_annotation_rows
)


print(
    "Combined annotation rows not found in raw NVIDIA data:",
    f"{len(unmatched_annotation_df):,}",
)


if len(unmatched_annotation_df) > 0:

    display(
        unmatched_annotation_df.head(
            20
        )
    )


assert len(unmatched_annotation_df) == 0, (
    f"{len(unmatched_annotation_df):,} rows in "
    f"{NVIDIA_TP_FILE.name} could not be matched "
    "to raw NVIDIA trajectory IDs."
)


,configuration,raw_annotation_rows,duplicate_rows,unique_annotation_ids,tp_ids,classification_counts
0,OpenHands + MiniMax-M2.5,1059,0,1059,1059,{'TEST_PANDERING': 1059}
1,OpenHands + Qwen3.5-122B,759,0,759,759,{'TEST_PANDERING': 759}
2,SWE-agent + MiniMax-M2.5,936,0,936,936,{'TEST_PANDERING': 936}
3,SWE-agent + Qwen3.5-122B,710,0,710,710,{'TEST_PANDERING': 710}


Combined annotation rows not found in raw NVIDIA data: 0


In [7]:
# ============================================================
# Cell 6B: Annotation population checkpoint
# ============================================================

total_raw_annotations = (
    annotation_summary[
        "raw_annotation_rows"
    ].sum()
)

total_duplicate_rows = (
    annotation_summary[
        "duplicate_rows"
    ].sum()
)

total_unique_tp = sum(
    len(tp_ids)
    for tp_ids in tp_sets.values()
)


print("=" * 100)
print("TP ANNOTATION CHECKPOINT")
print("=" * 100)

print(
    f"Raw annotation rows : "
    f"{total_raw_annotations:,}"
)

print(
    f"Duplicate rows      : "
    f"{total_duplicate_rows:,}"
)

print(
    f"Unique TP IDs       : "
    f"{total_unique_tp:,}"
)


TP ANNOTATION CHECKPOINT
Raw annotation rows : 3,464
Duplicate rows      : 0
Unique TP IDs       : 3,464


In [8]:
# ============================================================
# Cell 7: Align TP annotation IDs with raw Arrow trajectories
# ============================================================

alignment_rows = []


for config in NVIDIA_GROUPS:

    raw_config = (
        raw_df[
            raw_df[
                "configuration"
            ] == config
        ]
    )


    raw_ids = set(
        raw_config[
            "trajectory_id"
        ]
    )


    tp_ids = (
        tp_sets[
            config
        ]
    )


    matched_tp = (
        tp_ids
        & raw_ids
    )


    unmatched_tp = (
        tp_ids
        - raw_ids
    )


    alignment_rows.append(
        {
            "configuration":
                config,

            "raw_trajectories":
                len(raw_ids),

            "tp_annotations":
                len(tp_ids),

            "matched_tp":
                len(matched_tp),

            "unmatched_tp":
                len(unmatched_tp),

            "match_rate_pct":
                (
                    len(matched_tp)
                    / len(tp_ids)
                    * 100

                    if tp_ids

                    else np.nan
                ),
        }
    )


alignment_df = pd.DataFrame(
    alignment_rows
)


display(
    alignment_df.style.format(
        {
            "raw_trajectories":
                "{:,}",

            "tp_annotations":
                "{:,}",

            "matched_tp":
                "{:,}",

            "unmatched_tp":
                "{:,}",

            "match_rate_pct":
                "{:.2f}",
        }
    )
)


,configuration,raw_trajectories,tp_annotations,matched_tp,unmatched_tp,match_rate_pct
0,OpenHands + MiniMax-M2.5,"49,948","1,059","1,059",0,100.00
1,OpenHands + Qwen3.5-122B,"55,488",759,759,0,100.00
2,SWE-agent + MiniMax-M2.5,"57,268",936,936,0,100.00
3,SWE-agent + Qwen3.5-122B,"44,785",710,710,0,100.00


In [9]:
# ============================================================
# Cell 8: Build trajectory-level master dataset
# ============================================================

master_df = raw_df.copy()


# ------------------------------------------------------------
# Default:
# trajectory is not TP
# ------------------------------------------------------------

master_df[
    "tp"
] = 0


# ------------------------------------------------------------
# Mark canonical TP trajectories
# configuration-specific
# ------------------------------------------------------------

for config, tp_ids in tp_sets.items():

    mask = (
        master_df[
            "configuration"
        ].eq(config)
        &
        master_df[
            "trajectory_id"
        ].isin(
            tp_ids
        )
    )


    master_df.loc[
        mask,
        "tp"
    ] = 1


master_df[
    "tp"
] = (
    master_df[
        "tp"
    ]
    .astype("int8")
)


print("=" * 100)
print("MASTER TRAJECTORY DATA")
print("=" * 100)

print(
    f"Rows           : "
    f"{len(master_df):,}"
)

print(
    f"TP trajectories: "
    f"{master_df['tp'].sum():,}"
)


display(
    master_df.head()
)


MASTER TRAJECTORY DATA
Rows           : 207,489
TP trajectories: 3,464


,instance_id,trajectory_id,resolved,configuration,source_shard,agent,model,tp
0,python-attrs__attrs-770,f8853683-87bd-418b-b87f-19ab95c60317,1,OpenHands + MiniMax-M2.5,data-00000-of-00029.arrow,OpenHands,MiniMax,0
1,altair-viz__altair-3118,4003b9f9-720f-4897-a270-586055e35ab7,0,OpenHands + MiniMax-M2.5,data-00000-of-00029.arrow,OpenHands,MiniMax,0
2,open-telemetry__opentelemetry-go-contrib-6962,fe108abd-08de-44cb-b75c-1c9a5fd164d0,-1,OpenHands + MiniMax-M2.5,data-00000-of-00029.arrow,OpenHands,MiniMax,0
3,apache__dubbo-go-hessian2-212,702b0dab-5adb-481d-a410-27be367373f8,0,OpenHands + MiniMax-M2.5,data-00000-of-00029.arrow,OpenHands,MiniMax,0
4,alibaba__fescar-412,6922fc2a-9d34-4bfa-ab08-d46a37b5928b,0,OpenHands + MiniMax-M2.5,data-00000-of-00029.arrow,OpenHands,MiniMax,0


In [10]:
# ============================================================
# Cell 9: Resolved-status audit
# ============================================================

resolved_summary = (
    master_df
    .groupby(
        [
            "configuration",
            "resolved",
        ],
        as_index=False,
    )
    .agg(
        trajectories=(
            "trajectory_id",
            "size",
        ),

        tp_trajectories=(
            "tp",
            "sum",
        ),

        unique_prs=(
            "instance_id",
            "nunique",
        ),
    )
)


display(
    resolved_summary.style.format(
        {
            "trajectories":
                "{:,}",

            "tp_trajectories":
                "{:,}",

            "unique_prs":
                "{:,}",
        }
    )
)


,configuration,resolved,trajectories,tp_trajectories,unique_prs
0,OpenHands + MiniMax-M2.5,-1,"11,096",292,"4,644"
1,OpenHands + MiniMax-M2.5,0,"22,911",767,"10,292"
2,OpenHands + MiniMax-M2.5,1,"15,941",0,"6,993"
3,OpenHands + Qwen3.5-122B,-1,"12,684",246,"4,818"
4,OpenHands + Qwen3.5-122B,0,"29,147",513,"11,997"
5,OpenHands + Qwen3.5-122B,1,"13,657",0,"6,183"
6,SWE-agent + MiniMax-M2.5,-1,"13,163",213,"4,941"
7,SWE-agent + MiniMax-M2.5,0,"24,554",723,"10,268"
8,SWE-agent + MiniMax-M2.5,1,"19,551",0,"8,106"
9,SWE-agent + Qwen3.5-122B,-1,"9,815",199,"4,166"


## Table 1 - 1

In [11]:
# ============================================================
# Cell 10: Primary eligible trajectory population
#
# Main paper:
# resolved ∈ {0, -1}
# ============================================================

main_df = (
    master_df[
        master_df[
            "resolved"
        ].isin(
            [
                0,
                -1,
            ]
        )
    ]
    .copy()
)


main_summary = (
    main_df
    .groupby(
        "configuration",
        as_index=False,
    )
    .agg(
        eligible_trajectories=(
            "trajectory_id",
            "size",
        ),

        tp_trajectories=(
            "tp",
            "sum",
        ),

        eligible_prs=(
            "instance_id",
            "nunique",
        ),
    )
)


main_summary[
    "trajectory_tp_rate_pct"
] = (
    main_summary[
        "tp_trajectories"
    ]
    /
    main_summary[
        "eligible_trajectories"
    ]
    * 100
)


display(
    main_summary.style.format(
        {
            "eligible_trajectories":
                "{:,}",

            "tp_trajectories":
                "{:,}",

            "eligible_prs":
                "{:,}",

            "trajectory_tp_rate_pct":
                "{:.3f}",
        }
    )
)

,configuration,eligible_trajectories,tp_trajectories,eligible_prs,trajectory_tp_rate_pct
0,OpenHands + MiniMax-M2.5,"34,007","1,059","14,885",3.114
1,OpenHands + Qwen3.5-122B,"41,831",759,"16,752",1.814
2,SWE-agent + MiniMax-M2.5,"37,717",936,"15,164",2.482
3,SWE-agent + Qwen3.5-122B,"28,690",710,"12,977",2.475


In [12]:
# ============================================================
# Cell 11: PR-any binary TP outcome
# ============================================================

pr_any = (
    main_df
    .groupby(
        [
            "instance_id",
            "configuration",
        ],
        as_index=False,
    )
    .agg(
        # ----------------------------------------------------
        # PR/config is TP affected if ANY eligible trajectory
        # is TP
        # ----------------------------------------------------

        tp=(
            "tp",
            "max",
        ),

        n_attempts=(
            "trajectory_id",
            "size",
        ),

        n_tp_trajectories=(
            "tp",
            "sum",
        ),
    )
)


print("=" * 100)
print("PR-ANY POPULATION")
print("=" * 100)

print(
    f"PR/config rows : "
    f"{len(pr_any):,}"
)

print(
    f"Unique PRs     : "
    f"{pr_any['instance_id'].nunique():,}"
)


display(
    pr_any.head()
)


PR-ANY POPULATION
PR/config rows : 59,778
Unique PRs     : 19,441


,instance_id,configuration,tp,n_attempts,n_tp_trajectories
0,0xs34n__starknet.js-496,OpenHands + Qwen3.5-122B,0,1,0
1,0xs34n__starknet.js-508,OpenHands + MiniMax-M2.5,0,2,0
2,0xs34n__starknet.js-508,SWE-agent + Qwen3.5-122B,0,2,0
3,0xs34n__starknet.js-520,SWE-agent + MiniMax-M2.5,0,1,0
4,0xs34n__starknet.js-538,OpenHands + MiniMax-M2.5,1,2,1


## RQ2 Table 2 - 1
<a id="table-rq2-2-1"></a>

In [13]:
# ============================================================
# Cell 12: PR-level TP prevalence sanity check
# ============================================================

pr_summary = (
    pr_any
    .groupby(
        "configuration",
        as_index=False,
    )
    .agg(
        eligible_prs=(
            "instance_id",
            "size",
        ),

        tp_affected_prs=(
            "tp",
            "sum",
        ),
    )
)


pr_summary[
    "tp_rate_pct"
] = (
    pr_summary[
        "tp_affected_prs"
    ]
    /
    pr_summary[
        "eligible_prs"
    ]
    * 100
)


display(
    pr_summary.style.format(
        {
            "eligible_prs":
                "{:,}",

            "tp_affected_prs":
                "{:,}",

            "tp_rate_pct":
                "{:.2f}",
        }
    )
)

,configuration,eligible_prs,tp_affected_prs,tp_rate_pct
0,OpenHands + MiniMax-M2.5,"14,885",950,6.38
1,OpenHands + Qwen3.5-122B,"16,752",707,4.22
2,SWE-agent + MiniMax-M2.5,"15,164",819,5.40
3,SWE-agent + Qwen3.5-122B,"12,977",661,5.09


In [14]:
# ============================================================
# Cell 13: Define RQ2 pairwise comparisons
# ============================================================

OH_MINIMAX = "OpenHands + MiniMax-M2.5"
OH_QWEN = "OpenHands + Qwen3.5-122B"

SWE_MINIMAX = "SWE-agent + MiniMax-M2.5"
SWE_QWEN = "SWE-agent + Qwen3.5-122B"


PAIRWISE_COMPARISONS = {

    "Agent: OpenHands vs SWE-agent | MiniMax": (
        OH_MINIMAX,
        SWE_MINIMAX,
    ),

    "Agent: OpenHands vs SWE-agent | Qwen": (
        OH_QWEN,
        SWE_QWEN,
    ),

    "Model: MiniMax vs Qwen | OpenHands": (
        OH_MINIMAX,
        OH_QWEN,
    ),

    "Model: MiniMax vs Qwen | SWE-agent": (
        SWE_MINIMAX,
        SWE_QWEN,
    ),
}

In [15]:
# ============================================================
# Cell 14: Matched binary comparison helper
# ============================================================

def matched_binary_comparison(
    pr_outcomes,
    config_a,
    config_b,
):

    # --------------------------------------------------------
    # Outcomes for configuration A
    # --------------------------------------------------------

    a = (
        pr_outcomes[
            pr_outcomes[
                "configuration"
            ] == config_a
        ][
            [
                "instance_id",
                "tp",
            ]
        ]
        .rename(
            columns={
                "tp": "tp_a",
            }
        )
    )


    # --------------------------------------------------------
    # Outcomes for configuration B
    # --------------------------------------------------------

    b = (
        pr_outcomes[
            pr_outcomes[
                "configuration"
            ] == config_b
        ][
            [
                "instance_id",
                "tp",
            ]
        ]
        .rename(
            columns={
                "tp": "tp_b",
            }
        )
    )


    # --------------------------------------------------------
    # Inner join:
    #
    # PR must have >=1 eligible non-successful trajectory
    # under BOTH configurations
    # --------------------------------------------------------

    matched = (
        a.merge(
            b,
            on="instance_id",
            how="inner",
            validate="one_to_one",
        )
    )


    n = len(
        matched
    )


    # ========================================================
    # Matched 2x2 contingency table
    #
    #            B=0    B=1
    # A=0        n00    n01
    # A=1        n10    n11
    # ========================================================

    n11 = int(
        (
            matched["tp_a"].eq(1)
            &
            matched["tp_b"].eq(1)
        ).sum()
    )

    n10 = int(
        (
            matched["tp_a"].eq(1)
            &
            matched["tp_b"].eq(0)
        ).sum()
    )

    n01 = int(
        (
            matched["tp_a"].eq(0)
            &
            matched["tp_b"].eq(1)
        ).sum()
    )

    n00 = int(
        (
            matched["tp_a"].eq(0)
            &
            matched["tp_b"].eq(0)
        ).sum()
    )


    # ========================================================
    # TP rates
    # ========================================================

    rate_a = (
        (n11 + n10)
        / n
        * 100
    )

    rate_b = (
        (n11 + n01)
        / n
        * 100
    )


    # ========================================================
    # Matched odds ratio
    # ========================================================

    matched_or = (
        n10 / n01
        if n01 > 0
        else np.inf
    )


    # ========================================================
    # 95% Wald CI for matched OR
    # ========================================================

    if n10 > 0 and n01 > 0:

        log_or = math.log(
            matched_or
        )

        se_log_or = math.sqrt(
            1 / n10
            +
            1 / n01
        )

        ci_low = math.exp(
            log_or
            - 1.96 * se_log_or
        )

        ci_high = math.exp(
            log_or
            + 1.96 * se_log_or
        )

    else:

        ci_low = np.nan
        ci_high = np.nan


    # ========================================================
    # Exact two-sided McNemar test
    #
    # Conditional on discordant pairs:
    # n10 + n01 ~ Binomial(p=0.5)
    # ========================================================

    discordant = (
        n10
        + n01
    )


    if discordant > 0:

        p_value = binomtest(
            k=n10,
            n=discordant,
            p=0.5,
            alternative="two-sided",
        ).pvalue

    else:

        p_value = 1.0


    return {
        "matched_prs":
            n,

        "n11":
            n11,

        "n10":
            n10,

        "n01":
            n01,

        "n00":
            n00,

        "rate_a":
            rate_a,

        "rate_b":
            rate_b,

        "matched_or":
            matched_or,

        "ci_low":
            ci_low,

        "ci_high":
            ci_high,

        "p_value":
            p_value,
    }


In [16]:
# ============================================================
# Cell 15: Reproduce primary RQ2 pairwise results
# ============================================================

pairwise_rows = []


for comparison, (
    config_a,
    config_b,
) in PAIRWISE_COMPARISONS.items():

    result = matched_binary_comparison(
        pr_outcomes=pr_any,
        config_a=config_a,
        config_b=config_b,
    )


    pairwise_rows.append(
        {
            "comparison":
                comparison,

            "config_a":
                config_a,

            "config_b":
                config_b,

            **result,
        }
    )


main_pairwise = pd.DataFrame(
    pairwise_rows
)


display(
    main_pairwise.style.format(
        {
            "matched_prs":
                "{:,}",

            "n11":
                "{:,}",

            "n10":
                "{:,}",

            "n01":
                "{:,}",

            "n00":
                "{:,}",

            "rate_a":
                "{:.2f}",

            "rate_b":
                "{:.2f}",

            "matched_or":
                "{:.3f}",

            "ci_low":
                "{:.3f}",

            "ci_high":
                "{:.3f}",

            "p_value":
                "{:.3e}",
        }
    )
)


,comparison,config_a,config_b,matched_prs,n11,n10,n01,n00,rate_a,rate_b,matched_or,ci_low,ci_high,p_value
0,Agent: OpenHands vs SWE-agent | MiniMax,OpenHands + MiniMax-M2.5,SWE-agent + MiniMax-M2.5,"12,803",135,706,525,"11,437",6.57,5.16,1.345,1.201,1.506,2.760e-07
1,Agent: OpenHands vs SWE-agent | Qwen,OpenHands + Qwen3.5-122B,SWE-agent + Qwen3.5-122B,"11,899",53,480,570,"10,796",4.48,5.24,0.842,0.746,0.951,5.995e-03
2,Model: MiniMax vs Qwen | OpenHands,OpenHands + MiniMax-M2.5,OpenHands + Qwen3.5-122B,"13,291",63,787,473,"11,968",6.40,4.03,1.664,1.485,1.865,7.750e-19
3,Model: MiniMax vs Qwen | SWE-agent,SWE-agent + MiniMax-M2.5,SWE-agent + Qwen3.5-122B,"11,220",72,519,516,"10,113",5.27,5.24,1.006,0.890,1.136,9.504e-01


## Table 4
<a id="table-rq2-3"></a>

In [17]:
# ============================================================
# Cell 16: Paper-ready RQ2 pairwise table
# ============================================================

pairwise_paper = (
    main_pairwise[
        [
            "comparison",
            "matched_prs",
            "rate_a",
            "rate_b",
            "matched_or",
            "ci_low",
            "ci_high",
            "p_value",
        ]
    ]
    .copy()
)


display(
    pairwise_paper.style.format(
        {
            "matched_prs":
                "{:,}",

            "rate_a":
                "{:.2f}%",

            "rate_b":
                "{:.2f}%",

            "matched_or":
                "{:.3f}",

            "ci_low":
                "{:.3f}",

            "ci_high":
                "{:.3f}",

            "p_value":
                "{:.3e}",
        }
    )
)

,comparison,matched_prs,rate_a,rate_b,matched_or,ci_low,ci_high,p_value
0,Agent: OpenHands vs SWE-agent | MiniMax,"12,803",6.57%,5.16%,1.345,1.201,1.506,2.760e-07
1,Agent: OpenHands vs SWE-agent | Qwen,"11,899",4.48%,5.24%,0.842,0.746,0.951,5.995e-03
2,Model: MiniMax vs Qwen | OpenHands,"13,291",6.40%,4.03%,1.664,1.485,1.865,7.750e-19
3,Model: MiniMax vs Qwen | SWE-agent,"11,220",5.27%,5.24%,1.006,0.890,1.136,9.504e-01


In [19]:
# ============================================================
# Cell 17: Pairwise reproduction checkpoints
# ============================================================

EXPECTED_MATCHED_PRS = {
    "Agent: OpenHands vs SWE-agent | MiniMax":
        12803,

    "Agent: OpenHands vs SWE-agent | Qwen":
        11899,

    "Model: MiniMax vs Qwen | OpenHands":
        13291,

    "Model: MiniMax vs Qwen | SWE-agent":
        11220,
}


for row in (
    main_pairwise
    .itertuples(index=False)
):

    expected_n = (
        EXPECTED_MATCHED_PRS[
            row.comparison
        ]
    )

    assert (
        row.matched_prs
        == expected_n
    ), (
        f"{row.comparison}: "
        f"expected {expected_n:,}, "
        f"got {row.matched_prs:,}"
    )


print(
    "All four matched-PR population checks passed."
)

All four matched-PR population checks passed.


In [20]:
# ============================================================
# Cell 18: Verify matched OR from discordant cells
# ============================================================

for row in (
    main_pairwise
    .itertuples(index=False)
):

    assert (
        row.n00
        + row.n01
        + row.n10
        + row.n11
        == row.matched_prs
    )

    assert np.isclose(
        row.matched_or,
        row.n10 / row.n01,
    )


print(
    "All contingency-table and matched-OR checks passed."
)


display(
    main_pairwise[
        [
            "comparison",
            "matched_prs",
            "n11",
            "n10",
            "n01",
            "n00",
            "matched_or",
            "p_value",
        ]
    ]
)


All contingency-table and matched-OR checks passed.


,comparison,matched_prs,n11,n10,n01,n00,matched_or,p_value
0,Agent: OpenHands vs SWE-agent | MiniMax,12803,135,706,525,11437,1.344762,2.759535e-07
1,Agent: OpenHands vs SWE-agent | Qwen,11899,53,480,570,10796,0.842105,5.995459e-03
2,Model: MiniMax vs Qwen | OpenHands,13291,63,787,473,11968,1.663848,7.749647e-19
3,Model: MiniMax vs Qwen | SWE-agent,11220,72,519,516,10113,1.005814,9.504338e-01


In [21]:
# ============================================================
# Cell 19: Strict four-way agent/model coding
#
# Agent:
#   SWE-agent = 0
#   OpenHands = 1
#
# Model:
#   Qwen    = 0
#   MiniMax = 1
#
# Reference:
#   SWE-agent + Qwen
# ============================================================

STRICT_CONFIGS = {
    "OpenHands + MiniMax-M2.5": {
        "agent": 1,
        "model": 1,
    },

    "OpenHands + Qwen3.5-122B": {
        "agent": 1,
        "model": 0,
    },

    "SWE-agent + MiniMax-M2.5": {
        "agent": 0,
        "model": 1,
    },

    "SWE-agent + Qwen3.5-122B": {
        "agent": 0,
        "model": 0,
    },
}

In [22]:
# ============================================================
# Cell 20: Build strict four-way matched population
# ============================================================

def build_fourway_long(
    pr_outcomes
):

    required_configs = set(
        STRICT_CONFIGS.keys()
    )


    # --------------------------------------------------------
    # Configurations represented for each PR
    # --------------------------------------------------------

    config_sets = (
        pr_outcomes
        .groupby(
            "instance_id"
        )[
            "configuration"
        ]
        .agg(set)
    )


    # --------------------------------------------------------
    # Strict four-way:
    # PR must have eligible trajectories under ALL 4 configs
    # --------------------------------------------------------

    fourway_prs = set(
        config_sets[
            config_sets.apply(
                lambda configs:
                    required_configs.issubset(
                        configs
                    )
            )
        ].index
    )


    # --------------------------------------------------------
    # Keep exactly the 4 configuration outcomes per PR
    # --------------------------------------------------------

    long_df = (
        pr_outcomes[
            pr_outcomes[
                "instance_id"
            ].isin(
                fourway_prs
            )
            &
            pr_outcomes[
                "configuration"
            ].isin(
                required_configs
            )
        ][
            [
                "instance_id",
                "configuration",
                "tp",
            ]
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Add binary design variables
    # --------------------------------------------------------

    long_df[
        "agent"
    ] = (
        long_df[
            "configuration"
        ]
        .map(
            lambda config:
                STRICT_CONFIGS[
                    config
                ][
                    "agent"
                ]
        )
        .astype(int)
    )


    long_df[
        "model"
    ] = (
        long_df[
            "configuration"
        ]
        .map(
            lambda config:
                STRICT_CONFIGS[
                    config
                ][
                    "model"
                ]
        )
        .astype(int)
    )


    long_df[
        "agent_model"
    ] = (
        long_df[
            "agent"
        ]
        *
        long_df[
            "model"
        ]
    )


    # --------------------------------------------------------
    # Sanity checks:
    # exactly four rows per retained PR
    # --------------------------------------------------------

    rows_per_pr = (
        long_df
        .groupby(
            "instance_id"
        )
        .size()
    )


    assert (
        rows_per_pr.min()
        == 4
    )

    assert (
        rows_per_pr.max()
        == 4
    )

    assert (
        len(long_df)
        ==
        len(fourway_prs) * 4
    )


    return (
        long_df,
        fourway_prs,
    )


In [23]:
# ============================================================
# Cell 21: Strict four-way population checkpoint
# ============================================================

fourway_long, fourway_prs = (
    build_fourway_long(
        pr_any
    )
)


print("=" * 100)
print("STRICT FOUR-WAY MATCHED POPULATION")
print("=" * 100)

print(
    f"Four-way PRs : "
    f"{len(fourway_prs):,}"
)

print(
    f"Long rows    : "
    f"{len(fourway_long):,}"
)


display(
    fourway_long.head(
        12
    )
)


STRICT FOUR-WAY MATCHED POPULATION
Four-way PRs : 9,588
Long rows    : 38,352


,instance_id,configuration,tp,agent,model,agent_model
4,0xs34n__starknet.js-538,OpenHands + MiniMax-M2.5,1,1,1,1
5,0xs34n__starknet.js-538,OpenHands + Qwen3.5-122B,0,1,0,0
6,0xs34n__starknet.js-538,SWE-agent + MiniMax-M2.5,0,0,1,0
7,0xs34n__starknet.js-538,SWE-agent + Qwen3.5-122B,0,0,0,0
11,11ty__eleventy-1749,OpenHands + MiniMax-M2.5,0,1,1,1
12,11ty__eleventy-1749,OpenHands + Qwen3.5-122B,0,1,0,0
13,11ty__eleventy-1749,SWE-agent + MiniMax-M2.5,0,0,1,0
14,11ty__eleventy-1749,SWE-agent + Qwen3.5-122B,0,0,0,0
15,11ty__eleventy-1876,OpenHands + MiniMax-M2.5,0,1,1,1
16,11ty__eleventy-1876,OpenHands + Qwen3.5-122B,0,1,0,0


In [24]:
# ============================================================
# Cell 22: Identify informative PR strata
# ============================================================

pr_tp_sum = (
    fourway_long
    .groupby(
        "instance_id"
    )[
        "tp"
    ]
    .sum()
)


informative_prs = set(
    pr_tp_sum[
        (pr_tp_sum > 0)
        &
        (pr_tp_sum < 4)
    ].index
)


informative_long = (
    fourway_long[
        fourway_long[
            "instance_id"
        ].isin(
            informative_prs
        )
    ]
    .copy()
)


print("=" * 100)
print("INFORMATIVE FOUR-WAY PR STRATA")
print("=" * 100)

print(
    f"Four-way PRs      : "
    f"{len(fourway_prs):,}"
)

print(
    f"Informative PRs   : "
    f"{len(informative_prs):,}"
)

print(
    f"Model observations: "
    f"{len(informative_long):,}"
)


print(
    "\nTP-sum distribution across four-way PRs:"
)

display(
    pr_tp_sum
    .value_counts()
    .sort_index()
    .rename_axis(
        "n_tp_configurations"
    )
    .reset_index(
        name="n_prs"
    )
)

INFORMATIVE FOUR-WAY PR STRATA
Four-way PRs      : 9,588
Informative PRs   : 1,656
Model observations: 6,624

TP-sum distribution across four-way PRs:


,n_tp_configurations,n_prs
0,0,7929
1,1,1355
2,2,275
3,3,26
4,4,3


In [25]:
# ============================================================
# Cell 23: Fit conditional logistic regression
#
# logit P(TP_pc = 1)
# =
# alpha_p
# + beta_A * Agent
# + beta_M * Model
# + beta_AM * Agent*Model
#
# NO global intercept
# ============================================================

informative_long[
    "group_id"
], _ = pd.factorize(
    informative_long[
        "instance_id"
    ]
)


y = (
    informative_long[
        "tp"
    ]
    .astype(int)
)


X = (
    informative_long[
        [
            "agent",
            "model",
            "agent_model",
        ]
    ]
    .astype(float)
)


groups = (
    informative_long[
        "group_id"
    ]
)


conditional_model = ConditionalLogit(
    endog=y,
    exog=X,
    groups=groups,
)


conditional_result = (
    conditional_model.fit(
        method="bfgs",
        maxiter=1000,
        disp=False,
    )
)


print(
    conditional_result.summary()
)

                  Conditional Logit Model Regression Results                  
Dep. Variable:                     tp   No. Observations:                 6624
Model:               ConditionalLogit   No. groups:                       1656
Log-Likelihood:               -2385.1   Min group size:                      4
Method:                          bfgs   Max group size:                      4
Date:                Sun, 23 Aug 2026   Mean group size:                   4.0
Time:                        13:43:43                                         
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
agent          -0.2037      0.071     -2.865      0.004      -0.343      -0.064
model           0.0023      0.068      0.033      0.973      -0.131       0.136
agent_model     0.4454      0.097      4.611      0.000       0.256       0.635


In [26]:
# ============================================================
# Cell 24: Extract Agent × Model interaction
# ============================================================

params = (
    conditional_result.params
)

bse = (
    conditional_result.bse
)

pvalues = (
    conditional_result.pvalues
)

conf_int = (
    conditional_result.conf_int()
)


beta_interaction = (
    params[
        "agent_model"
    ]
)

se_interaction = (
    bse[
        "agent_model"
    ]
)

z_interaction = (
    beta_interaction
    / se_interaction
)

interaction_or = np.exp(
    beta_interaction
)

interaction_ci_low = np.exp(
    conf_int.loc[
        "agent_model",
        0,
    ]
)

interaction_ci_high = np.exp(
    conf_int.loc[
        "agent_model",
        1,
    ]
)

interaction_p = (
    pvalues[
        "agent_model"
    ]
)


interaction_summary = pd.DataFrame(
    [
        {
            "fourway_prs":
                len(
                    fourway_prs
                ),

            "informative_prs":
                len(
                    informative_prs
                ),

            "coef":
                beta_interaction,

            "std_err":
                se_interaction,

            "OR":
                interaction_or,

            "CI_low":
                interaction_ci_low,

            "CI_high":
                interaction_ci_high,

            "z":
                z_interaction,

            "p_value":
                interaction_p,
        }
    ]
)


display(
    interaction_summary.style.format(
        {
            "fourway_prs":
                "{:,}",

            "informative_prs":
                "{:,}",

            "coef":
                "{:.4f}",

            "std_err":
                "{:.4f}",

            "OR":
                "{:.3f}",

            "CI_low":
                "{:.3f}",

            "CI_high":
                "{:.3f}",

            "z":
                "{:.3f}",

            "p_value":
                "{:.3e}",
        }
    )
)


,fourway_prs,informative_prs,coef,std_err,OR,CI_low,CI_high,z,p_value
0,"9,588","1,656",0.4454,0.0966,1.561,1.292,1.886,4.611,4.001e-06


In [27]:
# ============================================================
# Cell 25: Interaction reproduction checkpoints
# ============================================================

assert (
    len(fourway_prs)
    == 9588
)

assert (
    len(informative_prs)
    == 1656
)

assert np.isclose(
    interaction_or,
    1.561,
    atol=0.002,
)

assert np.isclose(
    beta_interaction,
    0.4454,
    atol=0.002,
)

In [28]:
# ============================================================
# Cell 26: Linear contrast helper
# ============================================================

from scipy.stats import norm


cov = (
    conditional_result
    .cov_params()
)


def linear_contrast(
    weights,
):

    """
    weights:
        dict mapping coefficient name -> weight

    Example:
        beta_agent + beta_interaction

        {
            "agent": 1,
            "agent_model": 1
        }
    """

    coefficient_names = list(
        params.index
    )


    w = np.array(
        [
            weights.get(
                name,
                0.0,
            )
            for name
            in coefficient_names
        ],
        dtype=float,
    )


    beta_vector = (
        params[
            coefficient_names
        ]
        .to_numpy()
    )


    cov_matrix = (
        cov.loc[
            coefficient_names,
            coefficient_names,
        ]
        .to_numpy()
    )


    estimate = float(
        w @ beta_vector
    )


    variance = float(
        w
        @ cov_matrix
        @ w
    )


    std_err = math.sqrt(
        variance
    )


    z_value = (
        estimate
        / std_err
    )


    p_value = (
        2
        * norm.sf(
            abs(
                z_value
            )
        )
    )


    ci_low_beta = (
        estimate
        - 1.96 * std_err
    )

    ci_high_beta = (
        estimate
        + 1.96 * std_err
    )


    return {
        "coef":
            estimate,

        "std_err":
            std_err,

        "OR":
            math.exp(
                estimate
            ),

        "CI_low":
            math.exp(
                ci_low_beta
            ),

        "CI_high":
            math.exp(
                ci_high_beta
            ),

        "z":
            z_value,

        "p_value":
            p_value,
    }


In [29]:
# ============================================================
# Cell 27: Simple conditional effects
# ============================================================

conditional_effect_specs = [

    (
        "OpenHands vs SWE-agent | Qwen3.5-122B",
        {
            "agent": 1,
        },
    ),

    (
        "OpenHands vs SWE-agent | MiniMax-M2.5",
        {
            "agent": 1,
            "agent_model": 1,
        },
    ),

    (
        "MiniMax-M2.5 vs Qwen3.5-122B | SWE-agent",
        {
            "model": 1,
        },
    ),

    (
        "MiniMax-M2.5 vs Qwen3.5-122B | OpenHands",
        {
            "model": 1,
            "agent_model": 1,
        },
    ),
]


conditional_effect_rows = []


for comparison, weights in (
    conditional_effect_specs
):

    result = linear_contrast(
        weights
    )

    conditional_effect_rows.append(
        {
            "Comparison":
                comparison,

            **result,
        }
    )


conditional_effects = pd.DataFrame(
    conditional_effect_rows
)


display(
    conditional_effects.style.format(
        {
            "coef":
                "{:.4f}",

            "std_err":
                "{:.4f}",

            "OR":
                "{:.3f}",

            "CI_low":
                "{:.3f}",

            "CI_high":
                "{:.3f}",

            "z":
                "{:.3f}",

            "p_value":
                "{:.3e}",
        }
    )
)


,Comparison,coef,std_err,OR,CI_low,CI_high,z,p_value
0,OpenHands vs SWE-agent | Qwen3.5-122B,-0.2037,0.0711,0.816,0.710,0.938,-2.865,4.170e-03
1,OpenHands vs SWE-agent | MiniMax-M2.5,0.2417,0.0653,1.273,1.120,1.447,3.699,2.166e-04
2,MiniMax-M2.5 vs Qwen3.5-122B | SWE-agent,0.0023,0.0682,1.002,0.877,1.146,0.033,9.734e-01
3,MiniMax-M2.5 vs Qwen3.5-122B | OpenHands,0.4477,0.0684,1.565,1.368,1.789,6.543,6.016e-11


## Table 5
<a id="table-rq2-4"></a>

In [30]:
# ============================================================
# Cell 28: Complete paper-ready interaction table
# ============================================================

interaction_row = pd.DataFrame(
    [
        {
            "Comparison":
                "Agent × Model",

            "coef":
                beta_interaction,

            "std_err":
                se_interaction,

            "OR":
                interaction_or,

            "CI_low":
                interaction_ci_low,

            "CI_high":
                interaction_ci_high,

            "z":
                z_interaction,

            "p_value":
                interaction_p,
        }
    ]
)


rq2_interaction_table = pd.concat(
    [
        conditional_effects,
        interaction_row,
    ],
    ignore_index=True,
)


display(
    rq2_interaction_table.style.format(
        {
            "coef":
                "{:.4f}",

            "std_err":
                "{:.4f}",

            "OR":
                "{:.3f}",

            "CI_low":
                "{:.3f}",

            "CI_high":
                "{:.3f}",

            "z":
                "{:.3f}",

            "p_value":
                "{:.3e}",
        }
    )
)

,Comparison,coef,std_err,OR,CI_low,CI_high,z,p_value
0,OpenHands vs SWE-agent | Qwen3.5-122B,-0.2037,0.0711,0.816,0.710,0.938,-2.865,4.170e-03
1,OpenHands vs SWE-agent | MiniMax-M2.5,0.2417,0.0653,1.273,1.120,1.447,3.699,2.166e-04
2,MiniMax-M2.5 vs Qwen3.5-122B | SWE-agent,0.0023,0.0682,1.002,0.877,1.146,0.033,9.734e-01
3,MiniMax-M2.5 vs Qwen3.5-122B | OpenHands,0.4477,0.0684,1.565,1.368,1.789,6.543,6.016e-11
4,Agent × Model,0.4454,0.0966,1.561,1.292,1.886,4.611,4.001e-06


In [31]:
# ============================================================
# Cell 29: Reusable strict four-way interaction model
# ============================================================

def fit_interaction_model(
    pr_outcomes,
    analysis_name="analysis",
):

    # --------------------------------------------------------
    # Build strict four-way population
    # --------------------------------------------------------

    long_df, fourway_prs = (
        build_fourway_long(
            pr_outcomes
        )
    )


    # --------------------------------------------------------
    # Informative strata:
    #
    # TP sum = 0 -> all four configs NON-TP
    # TP sum = 4 -> all four configs TP
    #
    # Only sums 1, 2, 3 contribute information
    # --------------------------------------------------------

    pr_tp_sum = (
        long_df
        .groupby(
            "instance_id"
        )[
            "tp"
        ]
        .sum()
    )


    informative_prs = set(
        pr_tp_sum[
            (pr_tp_sum > 0)
            &
            (pr_tp_sum < 4)
        ].index
    )


    informative = (
        long_df[
            long_df[
                "instance_id"
            ].isin(
                informative_prs
            )
        ]
        .copy()
    )


    if len(informative_prs) == 0:

        raise ValueError(
            "No informative PR strata."
        )


    # --------------------------------------------------------
    # ConditionalLogit group IDs
    # --------------------------------------------------------

    informative[
        "group_id"
    ], _ = pd.factorize(
        informative[
            "instance_id"
        ]
    )


    y = (
        informative[
            "tp"
        ]
        .astype(int)
    )


    X = (
        informative[
            [
                "agent",
                "model",
                "agent_model",
            ]
        ]
        .astype(float)
    )


    groups = (
        informative[
            "group_id"
        ]
    )


    model = ConditionalLogit(
        endog=y,
        exog=X,
        groups=groups,
    )


    result = model.fit(
        method="bfgs",
        maxiter=1000,
        disp=False,
    )


    params = result.params
    bse = result.bse
    pvalues = result.pvalues
    conf_int = result.conf_int()


    beta_am = (
        params[
            "agent_model"
        ]
    )

    se_am = (
        bse[
            "agent_model"
        ]
    )


    output = {
        "analysis":
            analysis_name,

        "fourway_prs":
            len(
                fourway_prs
            ),

        "informative_prs":
            len(
                informative_prs
            ),

        "observations":
            len(
                informative
            ),

        "beta_interaction":
            beta_am,

        "se_interaction":
            se_am,

        "z_interaction":
            beta_am / se_am,

        "interaction_or":
            np.exp(
                beta_am
            ),

        "interaction_ci_low":
            np.exp(
                conf_int.loc[
                    "agent_model",
                    0,
                ]
            ),

        "interaction_ci_high":
            np.exp(
                conf_int.loc[
                    "agent_model",
                    1,
                ]
            ),

        "interaction_p":
            pvalues[
                "agent_model"
            ],
    }


    return (
        output,
        result,
        long_df,
    )


In [32]:
# ============================================================
# Cell 30: Primary interaction
#
# resolved ∈ {0, -1}
# PR-any
# ============================================================

primary_interaction, _, _ = (
    fit_interaction_model(
        pr_any,
        analysis_name="Primary PR-any",
    )
)


primary_interaction_df = pd.DataFrame(
    [
        primary_interaction
    ]
)


display(
    primary_interaction_df.style.format(
        {
            "fourway_prs":
                "{:,}",

            "informative_prs":
                "{:,}",

            "observations":
                "{:,}",

            "beta_interaction":
                "{:.4f}",

            "se_interaction":
                "{:.4f}",

            "z_interaction":
                "{:.3f}",

            "interaction_or":
                "{:.3f}",

            "interaction_ci_low":
                "{:.3f}",

            "interaction_ci_high":
                "{:.3f}",

            "interaction_p":
                "{:.3e}",
        }
    )
)

,analysis,fourway_prs,informative_prs,observations,beta_interaction,se_interaction,z_interaction,interaction_or,interaction_ci_low,interaction_ci_high,interaction_p
0,Primary PR-any,"9,588","1,656","6,624",0.4454,0.0966,4.611,1.561,1.292,1.886,4.001e-06


In [33]:
# ============================================================
# Cell 31: Explicit-failure-only population
#
# resolved == 0
# ============================================================

resolved0_df = (
    master_df[
        master_df[
            "resolved"
        ].eq(0)
    ]
    .copy()
)


resolved0_summary = (
    resolved0_df
    .groupby(
        "configuration",
        as_index=False,
    )
    .agg(
        failed_trajectories=(
            "trajectory_id",
            "size",
        ),

        tp_trajectories=(
            "tp",
            "sum",
        ),

        eligible_prs=(
            "instance_id",
            "nunique",
        ),
    )
)


display(
    resolved0_summary.style.format(
        {
            "failed_trajectories":
                "{:,}",

            "tp_trajectories":
                "{:,}",

            "eligible_prs":
                "{:,}",
        }
    )
)

,configuration,failed_trajectories,tp_trajectories,eligible_prs
0,OpenHands + MiniMax-M2.5,"22,911",767,"10,292"
1,OpenHands + Qwen3.5-122B,"29,147",513,"11,997"
2,SWE-agent + MiniMax-M2.5,"24,554",723,"10,268"
3,SWE-agent + Qwen3.5-122B,"18,875",511,"8,858"


In [34]:
# ============================================================
# Cell 32: Explicit failures + PR-any
# ============================================================

pr_any_resolved0 = (
    resolved0_df
    .groupby(
        [
            "instance_id",
            "configuration",
        ],
        as_index=False,
    )
    .agg(
        tp=(
            "tp",
            "max",
        ),

        n_attempts=(
            "trajectory_id",
            "size",
        ),

        n_tp_trajectories=(
            "tp",
            "sum",
        ),
    )
)


print(
    f"PR/config rows: "
    f"{len(pr_any_resolved0):,}"
)


display(
    pr_any_resolved0.head()
)

PR/config rows: 41,415


,instance_id,configuration,tp,n_attempts,n_tp_trajectories
0,0xs34n__starknet.js-496,OpenHands + Qwen3.5-122B,0,1,0
1,0xs34n__starknet.js-508,OpenHands + MiniMax-M2.5,0,2,0
2,0xs34n__starknet.js-508,SWE-agent + Qwen3.5-122B,0,2,0
3,0xs34n__starknet.js-520,SWE-agent + MiniMax-M2.5,0,1,0
4,0xs34n__starknet.js-538,OpenHands + MiniMax-M2.5,1,2,1


In [35]:
# ============================================================
# Cell 33: resolved=0, PR-any interaction
# ============================================================

resolved0_interaction, _, _ = (
    fit_interaction_model(
        pr_any_resolved0,
        analysis_name="resolved=0 PR-any",
    )
)


resolved0_interaction_df = pd.DataFrame(
    [
        resolved0_interaction
    ]
)


display(
    resolved0_interaction_df.style.format(
        {
            "fourway_prs":
                "{:,}",

            "informative_prs":
                "{:,}",

            "observations":
                "{:,}",

            "beta_interaction":
                "{:.4f}",

            "se_interaction":
                "{:.4f}",

            "interaction_or":
                "{:.3f}",

            "interaction_ci_low":
                "{:.3f}",

            "interaction_ci_high":
                "{:.3f}",

            "z_interaction":
                "{:.3f}",

            "interaction_p":
                "{:.3e}",
        }
    )
)


,analysis,fourway_prs,informative_prs,observations,beta_interaction,se_interaction,z_interaction,interaction_or,interaction_ci_low,interaction_ci_high,interaction_p
0,resolved=0 PR-any,"6,202","1,128","4,512",0.5120,0.1176,4.352,1.669,1.325,2.101,1.347e-05


In [36]:
# ============================================================
# Cell 34: Build k=1 sampling bases
# ============================================================

sample_base = (
    pr_any.copy()
)


sample_base[
    "sample_tp_probability"
] = (
    sample_base[
        "n_tp_trajectories"
    ]
    /
    sample_base[
        "n_attempts"
    ]
)


sample_base_resolved0 = (
    pr_any_resolved0.copy()
)


sample_base_resolved0[
    "sample_tp_probability"
] = (
    sample_base_resolved0[
        "n_tp_trajectories"
    ]
    /
    sample_base_resolved0[
        "n_attempts"
    ]
)


display(
    sample_base[
        [
            "instance_id",
            "configuration",
            "n_attempts",
            "n_tp_trajectories",
            "sample_tp_probability",
        ]
    ]
    .head(20)
)


,instance_id,configuration,n_attempts,n_tp_trajectories,sample_tp_probability
0,0xs34n__starknet.js-496,OpenHands + Qwen3.5-122B,1,0,0.000000
1,0xs34n__starknet.js-508,OpenHands + MiniMax-M2.5,2,0,0.000000
2,0xs34n__starknet.js-508,SWE-agent + Qwen3.5-122B,2,0,0.000000
3,0xs34n__starknet.js-520,SWE-agent + MiniMax-M2.5,1,0,0.000000
4,0xs34n__starknet.js-538,OpenHands + MiniMax-M2.5,2,1,0.500000
5,0xs34n__starknet.js-538,OpenHands + Qwen3.5-122B,3,0,0.000000
6,0xs34n__starknet.js-538,SWE-agent + MiniMax-M2.5,3,0,0.000000
7,0xs34n__starknet.js-538,SWE-agent + Qwen3.5-122B,3,0,0.000000
8,11ty__eleventy-120,OpenHands + MiniMax-M2.5,3,0,0.000000
9,11ty__eleventy-120,SWE-agent + MiniMax-M2.5,3,0,0.000000


In [37]:
# ============================================================
# Cell 35: Draw one k=1 PR-level outcome dataset
# ============================================================

def draw_k1_pr_outcomes(
    sample_base,
    rng,
):

    sampled = (
        sample_base[
            [
                "instance_id",
                "configuration",
                "sample_tp_probability",
            ]
        ]
        .copy()
    )


    sampled[
        "tp"
    ] = (
        rng.random(
            len(sampled)
        )
        <
        sampled[
            "sample_tp_probability"
        ].to_numpy()
    ).astype(
        "int8"
    )


    return sampled


In [38]:
# ============================================================
# Cell 37: Primary population, k=1
#
# resolved ∈ {0, -1}
# 1000 repetitions
# ============================================================

from tqdm.auto import tqdm


N_REPEATS = 1000
BASE_SEED = 20260822


rng_main_k1 = np.random.default_rng(
    BASE_SEED
)


main_k1_rows = []


for r in tqdm(
    range(N_REPEATS),
    desc="Primary k=1 interaction",
):

    sampled = (
        draw_k1_pr_outcomes(
            sample_base,
            rng_main_k1,
        )
    )


    try:

        result, _, _ = (
            fit_interaction_model(
                sampled,
                analysis_name="Primary k=1",
            )
        )


        result[
            "repeat"
        ] = r

        result[
            "converged"
        ] = True


        main_k1_rows.append(
            result
        )


    except Exception as e:

        main_k1_rows.append(
            {
                "repeat":
                    r,

                "analysis":
                    "Primary k=1",

                "converged":
                    False,

                "error":
                    str(e),
            }
        )


main_k1_runs = pd.DataFrame(
    main_k1_rows
)


print(
    "Successful fits:",
    int(
        main_k1_runs[
            "converged"
        ].sum()
    ),
    "/",
    N_REPEATS,
)


Primary k=1 interaction: 100%|██████████| 1000/1000 [15:09<00:00,  1.10it/s]

Successful fits: 1000 / 1000


In [39]:
# ============================================================
# Cell 38: Primary k=1 summary
# ============================================================

main_k1_ok = (
    main_k1_runs[
        main_k1_runs[
            "converged"
        ].eq(True)
    ]
    .copy()
)


main_k1_summary = pd.DataFrame(
    [
        {
            "Specification":
                "Equal exposure: r in {0,-1}, k=1",

            "Successful_fits":
                len(
                    main_k1_ok
                ),

            "Four_Way_PRs":
                main_k1_ok[
                    "fourway_prs"
                ].median(),

            "Informative_PRs":
                main_k1_ok[
                    "informative_prs"
                ].median(),

            "Interaction_OR":
                main_k1_ok[
                    "interaction_or"
                ].median(),

            "OR_q025":
                main_k1_ok[
                    "interaction_or"
                ].quantile(
                    0.025
                ),

            "OR_q975":
                main_k1_ok[
                    "interaction_or"
                ].quantile(
                    0.975
                ),

            "Significant_fraction":
                (
                    main_k1_ok[
                        "interaction_p"
                    ]
                    < 0.05
                ).mean(),

            "Positive_fraction":
                (
                    main_k1_ok[
                        "beta_interaction"
                    ]
                    > 0
                ).mean(),
        }
    ]
)


display(
    main_k1_summary.style.format(
        {
            "Successful_fits":
                "{:,}",

            "Four_Way_PRs":
                "{:,.0f}",

            "Informative_PRs":
                "{:,.0f}",

            "Interaction_OR":
                "{:.3f}",

            "OR_q025":
                "{:.3f}",

            "OR_q975":
                "{:.3f}",

            "Significant_fraction":
                "{:.3f}",

            "Positive_fraction":
                "{:.3f}",
        }
    )
)


,Specification,Successful_fits,Four_Way_PRs,Informative_PRs,Interaction_OR,OR_q025,OR_q975,Significant_fraction,Positive_fraction
0,"Equal exposure: r in {0,-1}, k=1","1,000","9,588",832,2.129,1.772,2.589,1.000,1.000


In [40]:
# ============================================================
# Cell 39: Explicit failures, k=1
#
# resolved = 0
# 1000 repetitions
# ============================================================

rng_resolved0_k1 = (
    np.random.default_rng(
        BASE_SEED
    )
)


resolved0_k1_rows = []


for r in tqdm(
    range(N_REPEATS),
    desc="resolved=0 + k=1 interaction",
):

    sampled = (
        draw_k1_pr_outcomes(
            sample_base_resolved0,
            rng_resolved0_k1,
        )
    )


    try:

        result, _, _ = (
            fit_interaction_model(
                sampled,
                analysis_name="resolved=0 + k=1",
            )
        )


        result[
            "repeat"
        ] = r

        result[
            "converged"
        ] = True


        resolved0_k1_rows.append(
            result
        )


    except Exception as e:

        resolved0_k1_rows.append(
            {
                "repeat":
                    r,

                "analysis":
                    "resolved=0 + k=1",

                "converged":
                    False,

                "error":
                    str(e),
            }
        )


resolved0_k1_runs = pd.DataFrame(
    resolved0_k1_rows
)


print(
    "Successful fits:",
    int(
        resolved0_k1_runs[
            "converged"
        ].sum()
    ),
    "/",
    N_REPEATS,
)

resolved=0 + k=1 interaction: 100%|██████████| 1000/1000 [09:35<00:00,  1.74it/s]

Successful fits: 1000 / 1000


In [41]:
# ============================================================
# Cell 40: resolved=0 + k=1 summary
# ============================================================

resolved0_k1_ok = (
    resolved0_k1_runs[
        resolved0_k1_runs[
            "converged"
        ].eq(True)
    ]
    .copy()
)


resolved0_k1_summary = pd.DataFrame(
    [
        {
            "Specification":
                "Explicit failures + equal exposure: r=0, k=1",

            "Successful_fits":
                len(
                    resolved0_k1_ok
                ),

            "Four_Way_PRs":
                resolved0_k1_ok[
                    "fourway_prs"
                ].median(),

            "Informative_PRs":
                resolved0_k1_ok[
                    "informative_prs"
                ].median(),

            "Interaction_OR":
                resolved0_k1_ok[
                    "interaction_or"
                ].median(),

            "OR_q025":
                resolved0_k1_ok[
                    "interaction_or"
                ].quantile(
                    0.025
                ),

            "OR_q975":
                resolved0_k1_ok[
                    "interaction_or"
                ].quantile(
                    0.975
                ),

            "Significant_fraction":
                (
                    resolved0_k1_ok[
                        "interaction_p"
                    ]
                    < 0.05
                ).mean(),

            "Positive_fraction":
                (
                    resolved0_k1_ok[
                        "beta_interaction"
                    ]
                    > 0
                ).mean(),
        }
    ]
)


display(
    resolved0_k1_summary.style.format(
        {
            "Successful_fits":
                "{:,}",

            "Four_Way_PRs":
                "{:,.0f}",

            "Informative_PRs":
                "{:,.0f}",

            "Interaction_OR":
                "{:.3f}",

            "OR_q025":
                "{:.3f}",

            "OR_q975":
                "{:.3f}",

            "Significant_fraction":
                "{:.3f}",

            "Positive_fraction":
                "{:.3f}",
        }
    )
)


,Specification,Successful_fits,Four_Way_PRs,Informative_PRs,Interaction_OR,OR_q025,OR_q975,Significant_fraction,Positive_fraction
0,"Explicit failures + equal exposure: r=0, k=1","1,000","6,202",587,2.419,1.920,3.008,1.000,1.000


## Table 6
<a id="table-rq2-5"></a>

In [42]:
# ============================================================
# Cell 41: Final RQ2 sensitivity table
# ============================================================

rq2_sensitivity_table = pd.DataFrame(
    [
        {
            "Specification":
                "Primary: r in {0,-1}, PR-any",

            "Four_Way_PRs":
                primary_interaction[
                    "fourway_prs"
                ],

            "Informative_PRs":
                primary_interaction[
                    "informative_prs"
                ],

            "Interaction_OR":
                primary_interaction[
                    "interaction_or"
                ],

            "Interval_Low":
                primary_interaction[
                    "interaction_ci_low"
                ],

            "Interval_High":
                primary_interaction[
                    "interaction_ci_high"
                ],

            "Significance":
                (
                    f"p={primary_interaction['interaction_p']:.3e}"
                ),
        },

        {
            "Specification":
                "Equal exposure: r in {0,-1}, k=1",

            "Four_Way_PRs":
                int(
                    main_k1_ok[
                        "fourway_prs"
                    ].median()
                ),

            "Informative_PRs":
                int(
                    main_k1_ok[
                        "informative_prs"
                    ].median()
                ),

            "Interaction_OR":
                main_k1_ok[
                    "interaction_or"
                ].median(),

            "Interval_Low":
                main_k1_ok[
                    "interaction_or"
                ].quantile(
                    0.025
                ),

            "Interval_High":
                main_k1_ok[
                    "interaction_or"
                ].quantile(
                    0.975
                ),

            "Significance":
                (
                    f"{(main_k1_ok['interaction_p'] < 0.05).mean() * 100:.0f}% significant"
                ),
        },

        {
            "Specification":
                "Explicit failures: r=0, PR-any",

            "Four_Way_PRs":
                resolved0_interaction[
                    "fourway_prs"
                ],

            "Informative_PRs":
                resolved0_interaction[
                    "informative_prs"
                ],

            "Interaction_OR":
                resolved0_interaction[
                    "interaction_or"
                ],

            "Interval_Low":
                resolved0_interaction[
                    "interaction_ci_low"
                ],

            "Interval_High":
                resolved0_interaction[
                    "interaction_ci_high"
                ],

            "Significance":
                (
                    f"p={resolved0_interaction['interaction_p']:.3e}"
                ),
        },

        {
            "Specification":
                "Explicit failures + equal exposure: r=0, k=1",

            "Four_Way_PRs":
                int(
                    resolved0_k1_ok[
                        "fourway_prs"
                    ].median()
                ),

            "Informative_PRs":
                int(
                    resolved0_k1_ok[
                        "informative_prs"
                    ].median()
                ),

            "Interaction_OR":
                resolved0_k1_ok[
                    "interaction_or"
                ].median(),

            "Interval_Low":
                resolved0_k1_ok[
                    "interaction_or"
                ].quantile(
                    0.025
                ),

            "Interval_High":
                resolved0_k1_ok[
                    "interaction_or"
                ].quantile(
                    0.975
                ),

            "Significance":
                (
                    f"{(resolved0_k1_ok['interaction_p'] < 0.05).mean() * 100:.0f}% significant"
                ),
        },
    ]
)


display(
    rq2_sensitivity_table.style.format(
        {
            "Four_Way_PRs":
                "{:,}",

            "Informative_PRs":
                "{:,}",

            "Interaction_OR":
                "{:.3f}",

            "Interval_Low":
                "{:.3f}",

            "Interval_High":
                "{:.3f}",
        }
    )
)

,Specification,Four_Way_PRs,Informative_PRs,Interaction_OR,Interval_Low,Interval_High,Significance
0,"Primary: r in {0,-1}, PR-any","9,588","1,656",1.561,1.292,1.886,p=4.001e-06
1,"Equal exposure: r in {0,-1}, k=1","9,588",832,2.129,1.772,2.589,100% significant
2,"Explicit failures: r=0, PR-any","6,202","1,128",1.669,1.325,2.101,p=1.347e-05
3,"Explicit failures + equal exposure: r=0, k=1","6,202",587,2.419,1.920,3.008,100% significant


 ============================================================ <br>
 RQ1 Robustness — Nebius equal-exposure analysis (k = 1)<br><br>

 Purpose:<br>
   Compare conventional PR-any prevalence with prevalence under<br>
   equalized one-attempt exposure.<br><br>

 Method:<br>
   For each model and each repetition:<br>
     1. Randomly select one non-successful trajectory per PR.<br>
     2. Determine whether the selected trajectory is TP.<br>
     3. Calculate the percentage of PRs whose selected trajectory<br>
        is classified as TEST_PANDERING.<br><br>

   Repeat 1,000 times and report:<br>
     - median<br>
     - 2.5th percentile<br>
     - 97.5th percentile<br>
 ============================================================<br>

In [ ]:
# ============================================================
# Nebius Robustness Analysis
# PR-any vs k=1 Equal-Exposure
#
# Standalone version:
#   - does NOT require nebius_failed_df
#   - does NOT require nebius_k1_trajectories
#   - reconstructs the population directly from raw Nebius data
#   - uses exact model + trajectory linkage for TP annotations
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd

from datasets import load_dataset


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/home/chufeng/Desktop/CSI/ICSE2027"
)

TP_ROOT = (
    PROJECT_ROOT
    / "Data Preparation"
    / "data"
)

NEBIUS_ANNOTATION_FILE = (
    TP_ROOT
    / "nebius-swe_final.jsonl"
)

NEBIUS_DATASET = (
    "nebius/SWE-agent-trajectories"
)


TP_LABEL = "TEST_PANDERING"

K1_REPLICATIONS = 1_000
K1_RANDOM_SEED = 20260822


NEBIUS_RAW_MODEL_MAP = {
    "swe-agent-llama-405b": "Llama-405B",
    "swe-agent-llama-70b": "Llama-70B",
    "swe-agent-llama-8b": "Llama-8B",
}


NEBIUS_ANNOTATION_MODEL_MAP = {
    "swe-agent-llama-405b": "Llama-405B",
    "swe-agent-llama-70b": "Llama-70B",
    "swe-agent-llama-8b": "Llama-8B",

    "llama-405b": "Llama-405B",
    "llama-70b": "Llama-70B",
    "llama-8b": "Llama-8B",
}


# ------------------------------------------------------------
# 2. Validate annotation file
# ------------------------------------------------------------

assert NEBIUS_ANNOTATION_FILE.exists(), (
    "Nebius annotation file does not exist:\n"
    f"{NEBIUS_ANNOTATION_FILE}"
)


# ------------------------------------------------------------
# 3. Load and canonicalize Nebius annotations
#
# Key:
#   (canonical model, trajectory_id)
#
# Last occurrence wins.
# ------------------------------------------------------------

nebius_annotation_canonical = {}

raw_annotation_rows = 0
duplicate_annotation_rows = 0


with NEBIUS_ANNOTATION_FILE.open(
    "r",
    encoding="utf-8",
) as f:

    for line_number, line in enumerate(
        f,
        start=1,
    ):

        line = line.strip()

        if not line:
            continue


        record = json.loads(line)

        trajectory_id = record.get(
            "trajectory_id"
        )

        if trajectory_id is None:
            continue


        trajectory_id = str(
            trajectory_id
        ).strip()


        raw_model = str(
            record.get(
                "model_name",
                "",
            )
        ).strip().lower()


        model = (
            NEBIUS_ANNOTATION_MODEL_MAP.get(
                raw_model
            )
        )


        if model is None:

            raise ValueError(
                "Unknown Nebius annotation model:\n"
                f"line={line_number}\n"
                f"model_name={raw_model!r}"
            )


        key = (
            model,
            trajectory_id,
        )


        raw_annotation_rows += 1


        if key in nebius_annotation_canonical:

            duplicate_annotation_rows += 1


        # Last occurrence wins
        nebius_annotation_canonical[
            key
        ] = record


print("=" * 100)
print("NEBIUS ANNOTATIONS")
print("=" * 100)

print(
    "Raw annotation rows:",
    f"{raw_annotation_rows:,}",
)

print(
    "Duplicate annotation rows:",
    f"{duplicate_annotation_rows:,}",
)

print(
    "Canonical annotation rows:",
    f"{len(nebius_annotation_canonical):,}",
)


# ------------------------------------------------------------
# 4. Load raw Nebius dataset
# ------------------------------------------------------------

dataset = load_dataset(
    NEBIUS_DATASET,
    split="train",
    streaming=True,
)


# ------------------------------------------------------------
# 5. Reconstruct all non-successful trajectories
#
# Important:
#
# trajectory_id in the annotation file is reconstructed as
#
#     {instance_id}_{global_index}
#
# We require BOTH:
#
#     raw model == annotation model
#     raw trajectory_id == annotation trajectory_id
#
# Therefore model-assignment conflicts are excluded from TP.
# ------------------------------------------------------------

trajectory_rows = []

matched_tp_keys = set()


raw_trajectory_count = 0
failed_trajectory_count = 0


for global_index, raw_record in enumerate(
    dataset
):

    raw_trajectory_count += 1


    raw_model_name = str(
        raw_record[
            "model_name"
        ]
    ).strip().lower()


    canonical_model = (
        NEBIUS_RAW_MODEL_MAP.get(
            raw_model_name
        )
    )


    if canonical_model is None:

        raise ValueError(
            "Unknown raw Nebius model:\n"
            f"{raw_model_name!r}"
        )


    instance_id = str(
        raw_record[
            "instance_id"
        ]
    ).strip()


    trajectory_id = (
        f"{instance_id}_{global_index}"
    )


    # --------------------------------------------------------
    # Only non-successful trajectories belong to the analysis.
    # --------------------------------------------------------

    if raw_record[
        "target"
    ] is not False:

        continue


    failed_trajectory_count += 1


    annotation_key = (
        canonical_model,
        trajectory_id,
    )


    annotation = (
        nebius_annotation_canonical.get(
            annotation_key
        )
    )


    is_tp = int(
        annotation is not None
        and str(
            annotation.get(
                "classification",
                "",
            )
        ).strip().upper()
        == TP_LABEL
    )


    if is_tp:

        matched_tp_keys.add(
            annotation_key
        )


    trajectory_rows.append(
        {
            "model":
                canonical_model,

            "pr_id":
                instance_id,

            "trajectory_id":
                trajectory_id,

            "is_tp":
                is_tp,
        }
    )


nebius_k1_trajectories = pd.DataFrame(
    trajectory_rows
)


# ------------------------------------------------------------
# 6. Identify TP annotations excluded because exact model +
# trajectory linkage failed
# ------------------------------------------------------------

all_annotation_tp_keys = {
    key

    for key, record
    in nebius_annotation_canonical.items()

    if str(
        record.get(
            "classification",
            "",
        )
    ).strip().upper()
    == TP_LABEL
}


excluded_tp_keys = (
    all_annotation_tp_keys
    - matched_tp_keys
)


# ------------------------------------------------------------
# 7. Population validation
# ------------------------------------------------------------

assert (
    raw_trajectory_count
    == 80_036
), (
    "Unexpected raw Nebius trajectory count: "
    f"{raw_trajectory_count:,}"
)


assert (
    failed_trajectory_count
    == 66_647
), (
    "Unexpected Nebius non-successful trajectory count: "
    f"{failed_trajectory_count:,}"
)


assert (
    len(nebius_k1_trajectories)
    == 66_647
)


print()
print("=" * 100)
print("EXACT-LINKAGE k=1 POPULATION")
print("=" * 100)

print(
    "Raw trajectories:",
    f"{raw_trajectory_count:,}",
)

print(
    "Non-successful trajectories:",
    f"{failed_trajectory_count:,}",
)

print(
    "Exact-linkage TP trajectories:",
    f"{len(matched_tp_keys):,}",
)

print(
    "Excluded TP annotations:",
    f"{len(excluded_tp_keys):,}",
)


# ------------------------------------------------------------
# 8. Aggregate trajectory exposure by model and PR
# ------------------------------------------------------------

nebius_pr_exposure = (
    nebius_k1_trajectories
    .groupby(
        [
            "model",
            "pr_id",
        ],
        as_index=False,
    )
    .agg(
        n_non_successful=(
            "trajectory_id",
            "size",
        ),

        n_tp=(
            "is_tp",
            "sum",
        ),
    )
)


assert (
    nebius_pr_exposure[
        "n_tp"
    ]
    <=
    nebius_pr_exposure[
        "n_non_successful"
    ]
).all()


nebius_pr_exposure[
    "pr_any_tp"
] = (
    nebius_pr_exposure[
        "n_tp"
    ] > 0
).astype(np.int8)


# ------------------------------------------------------------
# 9. PR-any prevalence
# ------------------------------------------------------------

pr_any_summary = (
    nebius_pr_exposure
    .groupby(
        "model",
        as_index=False,
    )
    .agg(
        n_prs=(
            "pr_id",
            "size",
        ),

        n_non_successful=(
            "n_non_successful",
            "sum",
        ),

        n_tp=(
            "n_tp",
            "sum",
        ),

        tp_affected_prs=(
            "pr_any_tp",
            "sum",
        ),

        median_attempts_per_pr=(
            "n_non_successful",
            "median",
        ),
    )
)


pr_any_summary[
    "pr_any_pct"
] = (
    pr_any_summary[
        "tp_affected_prs"
    ]
    /
    pr_any_summary[
        "n_prs"
    ]
    * 100
)


pr_any_summary[
    "trajectory_level_pct"
] = (
    pr_any_summary[
        "n_tp"
    ]
    /
    pr_any_summary[
        "n_non_successful"
    ]
    * 100
)


# ------------------------------------------------------------
# 10. k=1 equal-exposure simulation
#
# For every PR, select exactly ONE uniformly random
# non-successful trajectory.
#
# Probability that sampled trajectory is TP:
#
#     n_tp / n_non_successful
#
# This is mathematically equivalent to explicitly drawing
# one trajectory from each PR.
# ------------------------------------------------------------

rng = np.random.default_rng(
    K1_RANDOM_SEED
)


replication_rows = []


for model, model_df in (
    nebius_pr_exposure
    .groupby(
        "model",
        sort=True,
    )
):

    tp_probability = (
        model_df[
            "n_tp"
        ].to_numpy(
            dtype=float
        )
        /
        model_df[
            "n_non_successful"
        ].to_numpy(
            dtype=float
        )
    )


    n_prs = len(
        tp_probability
    )


    sampled_tp = (
        rng.random(
            size=(
                K1_REPLICATIONS,
                n_prs,
            )
        )
        <
        tp_probability
    )


    prevalence = (
        sampled_tp.mean(
            axis=1
        )
        * 100
    )


    replication_rows.extend(
        {
            "model":
                model,

            "replication":
                repetition + 1,

            "k1_prevalence_pct":
                value,
        }

        for repetition, value
        in enumerate(
            prevalence
        )
    )


nebius_k1_replications = pd.DataFrame(
    replication_rows
)


# ------------------------------------------------------------
# 11. Summarize k=1 distribution
# ------------------------------------------------------------

k1_summary = (
    nebius_k1_replications
    .groupby(
        "model",
        as_index=False,
    )
    .agg(
        k1_median_pct=(
            "k1_prevalence_pct",
            "median",
        ),

        k1_p2_5_pct=(
            "k1_prevalence_pct",
            lambda x:
                np.percentile(
                    x,
                    2.5,
                ),
        ),

        k1_p97_5_pct=(
            "k1_prevalence_pct",
            lambda x:
                np.percentile(
                    x,
                    97.5,
                ),
        ),

        k1_mean_pct=(
            "k1_prevalence_pct",
            "mean",
        ),
    )
)


# ------------------------------------------------------------
# 12. Final combined table
# ------------------------------------------------------------

nebius_exposure_summary = (
    pr_any_summary
    .merge(
        k1_summary,
        on="model",
        how="left",
        validate="one_to_one",
    )
    .rename(
        columns={
            "model":
                "Model",

            "n_prs":
                "PRs",

            "n_non_successful":
                "Non-successful Traj.",

            "n_tp":
                "TP Traj.",

            "tp_affected_prs":
                "PR-any TP PRs",

            "pr_any_pct":
                "PR-any (%)",

            "trajectory_level_pct":
                "Trajectory TP (%)",

            "median_attempts_per_pr":
                "Median attempts/PR",

            "k1_median_pct":
                "k=1 Median (%)",

            "k1_p2_5_pct":
                "k=1 2.5th (%)",

            "k1_p97_5_pct":
                "k=1 97.5th (%)",

            "k1_mean_pct":
                "k=1 Mean (%)",
        }
    )
)


percentage_columns = [
    "PR-any (%)",
    "Trajectory TP (%)",
    "k=1 Median (%)",
    "k=1 2.5th (%)",
    "k=1 97.5th (%)",
    "k=1 Mean (%)",
]


nebius_exposure_summary[
    percentage_columns
] = (
    nebius_exposure_summary[
        percentage_columns
    ]
    .round(2)
)


display(
    nebius_exposure_summary
)


print()
print(
    f"✓ Completed "
    f"{K1_REPLICATIONS:,} "
    "k=1 repetitions per Nebius model."
)

NEBIUS ANNOTATIONS
Raw annotation rows: 9,320
Duplicate annotation rows: 0
Canonical annotation rows: 9,320

EXACT-LINKAGE k=1 POPULATION
Raw trajectories: 80,036
Non-successful trajectories: 66,647
Exact-linkage TP trajectories: 9,294
Excluded TP annotations: 26


,Model,PRs,Non-successful Traj.,TP Traj.,PR-any TP PRs,Median attempts/PR,PR-any (%),Trajectory TP (%),k=1 Median (%),k=1 2.5th (%),k=1 97.5th (%),k=1 Mean (%)
0,Llama-405B,98,883,102,45,10.0,45.92,11.55,11.22,6.12,16.33,11.34
1,Llama-70B,3555,62325,8840,2195,13.0,61.74,14.18,13.92,12.88,14.85,13.90
2,Llama-8B,523,3439,352,192,5.0,36.71,10.24,10.90,8.60,13.19,10.87



✓ Completed 1,000 k=1 repetitions per Nebius model.


: 